In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import matplotlib.pyplot as plt
import nichepca as npc
import scanpy as sc

from spatial_tcr.spatial import annotate_ccs, merge_labels

figure_dir = "figures/"

In [ ]:
path = "data/xenium/processed/01-kidney_tcr_qc.h5ad"
adata = sc.read_h5ad(path)
adata

In [ ]:
npc.gc.construct_multi_sample_graph(
    adata, sample_key="sample", radius=75, remove_self_loops=True
)

In [ ]:
adata

In [ ]:
annotate_ccs(adata)

In [ ]:
for s in adata.obs["sample"].unique():
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(ad_sub, color="cc", spot_size=10)
    break

In [ ]:
adata.obs.cc.value_counts().plot(kind="bar", logy=True)

## Merge clusters if necessary

In [ ]:
samples = adata.obs["sample"].unique().tolist()
for s in samples[0::]:
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(ad_sub, color="cc", spot_size=10)
    # break

In [ ]:
merge_labels(adata, labels=["0", "2"], label_key="cc")

In [ ]:
samples = adata.obs["sample"].unique().tolist()
for s in samples[0:1]:
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(ad_sub, color="cc", spot_size=10)
    break

In [ ]:
adata = adata[adata.obs["cc"] != "NA"].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
adata.obs.cc.value_counts().plot(kind="bar", logy=True, ax=ax)
ax.set_title("Cluster counts")
ax.set_xlabel("Cluster")
ax.set_ylabel("Count")
plt.show()

In [ ]:
def cc_to_sample(adata, cc):
    return adata.obs.loc[adata.obs["cc"] == cc, "sample"].iloc[0]


s = cc_to_sample(adata, "47")
ad_sub = adata[adata.obs["sample"] == s].copy()
sc.pl.spatial(ad_sub, color="cc", spot_size=10)

In [ ]:
min_size = 1000
mask = adata.obs.cc.value_counts() > min_size
retained_ccs = mask[mask].index.tolist()
adata = adata[adata.obs["cc"].isin(retained_ccs)].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
adata.obs.cc.value_counts().plot(kind="bar", logy=True, ax=ax)
ax.set_title("Cluster counts")
ax.set_xlabel("Cluster")
ax.set_ylabel("Count")
plt.show()

In [ ]:
samples = adata.obs["sample"].unique().tolist()
for s in samples[4:6]:
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(ad_sub, color="cc", spot_size=10)
    # break

In [ ]:
adata

In [ ]:
adata.write_h5ad("data/xenium/processed/02-kidney_tcr_split.h5ad")